# Ứng dụng 2: Mẫu trực quan trên ảnh (Caltech-101)

**Câu hỏi:** mỗi loại vật thể có những tổ hợp chi tiết hình ảnh nào đặc trưng, và các tổ hợp đó có giúp phân loại ảnh không?

Notebook này chỉ **điều phối**: mỗi bước là một file trong `applications/app2_image/src/`, ghi kết quả trung gian ra `applications/app2_image/data/` và `applications/app2_image/out/`.
Nếu Colab ngắt giữa chừng, chạy lại cell thiết lập rồi chạy tiếp từ bước đang dở (các bước bỏ qua phần đã làm xong).

| Bước | File | Kết quả |
|---|---|---|
| 1 | `s01_prepare_data.py` | `data/images/`, `data/split.csv` |
| 1b | `s02_sift_probe.py` | `out/sift_probe.csv`, `out/sift_probe.png` |
| 2 | `s03_sift_extract.py` | `data/sift.npz`, `out/sift_stats.csv` |
| 3–7 | `s04` … `s08` | *(đang làm)* |

## Thư viện sử dụng

Theo hướng dẫn của thầy, phần tiền xử lý được dùng thư viện hỗ trợ. **Khai thác tập phổ biến chỉ dùng FPClose do nhóm tự cài đặt (Julia)**; không dùng mlxtend, efficient-apriori hay thư viện FIM nào khác. SPMF chỉ dùng để đối chiếu kết quả.

**Tiền xử lý (biến ảnh thành giao dịch)**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| OpenCV (`opencv-python-headless`) | Đọc ảnh, chuyển xám, resize; trích đặc trưng SIFT | 1, 2 |
| NumPy | Mảng descriptor, tọa độ keypoint, histogram | 2–4 |
| scikit-learn | Chia train/test theo lớp (`train_test_split`); gom cụm tạo codebook (`MiniBatchKMeans`) | 1, 3 |
| pandas | Danh sách ảnh, bảng thống kê | 1–4 |

**Khai thác:** FPClose của nhóm (`demo.jl`), bước 5.

**Phân tích và đánh giá sau khai thác**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| pandas | Bảng mẫu, growth rate, bảng kết quả | 6, 7 |
| Matplotlib | Vẽ keypoint và mẫu lên ảnh, biểu đồ | 1b, 6, 7 |
| scikit-learn | Phân loại (`LogisticRegression`), accuracy, macro-F1, confusion matrix | 7 |


## 0. Thiết lập môi trường

In [ ]:
import os, sys, subprocess
from pathlib import Path

USE_OWN_IMPL = True      # True: khai thác bằng FPClose (Julia) của nhóm. Bản nộp PHẢI là True.
REPO_URL = "https://github.com/DRAGON-1312/FPClose.git"
REPO_BRANCH = "main"     # muốn thử trên Colab trước khi merge: đổi thành nhánh đang làm, vd "feature/app2"
APP_REL = Path("applications/app2_image")
IN_COLAB = "google.colab" in sys.modules

def find_repo_dir():
    """Gốc repo FPClose: đi từ thư mục hiện tại lên trên, tìm thư mục có applications/app2_image."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / APP_REL / "src" / "s01_prepare_data.py").exists():
            return d
    return None

REPO_DIR = find_repo_dir()
if REPO_DIR is None and IN_COLAB:                  # Colab: clone repo về /content/FPClose (một lần)
    REPO_DIR = Path("/content/FPClose")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
assert REPO_DIR is not None and (REPO_DIR / APP_REL / "src").is_dir(), \
    "Không tìm thấy applications/app2_image: mở notebook từ trong repo FPClose, hoặc chạy trên Colab"
APP_DIR = REPO_DIR / APP_REL

os.chdir(APP_DIR)                                  # các bước dùng đường dẫn tương đối src/, data/, out/
os.environ["PYTHONIOENCODING"] = "utf-8"           # để in tiếng Việt trên console Windows
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(APP_DIR / "src") not in sys.path:
    sys.path.insert(0, str(APP_DIR / "src"))

def run_step(script, *args):
    """Chạy một bước trong src/ bằng đúng Python của kernel, in output, dừng nếu lỗi."""
    r = subprocess.run([sys.executable, f"src/{script}", *map(str, args)],
                       capture_output=True, text=True, encoding="utf-8")
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} lỗi (mã {r.returncode})")

import config
print("REPO_DIR =", REPO_DIR, "| Colab:", IN_COLAB, "| SEED =", config.SEED)

## 1. Chuẩn bị dữ liệu

- **5 lớp** `airplanes`, `Motorbikes`, `Faces_easy`, `watch`, `Leopards`: hình dạng khác nhau rõ, đủ ảnh. Bỏ `Faces` (trùng `Faces_easy`) và `BACKGROUND_Google` (không phải một loại vật thể).
- **200 ảnh/lớp**: `Leopards` chỉ có 200 ảnh, nên đây là mức cân bằng lớn nhất. Lớp cân bằng thì support giữa các lớp mới so sánh được (growth rate).
- **Train/test 70/30 theo từng lớp** (140/60), seed 42. Mẫu chỉ khai thác trên train.
- **Ảnh xám, cạnh dài 400 px**: SIFT chỉ dùng độ sáng; cùng kích thước để số keypoint không phụ thuộc độ phân giải gốc.

In [ ]:
run_step("s01_prepare_data.py")

In [ ]:
import pandas as pd
split = pd.read_csv(config.DATA_SPLIT)
pd.crosstab(split.label, split.split, margins=True)

## 1b. Chạy thử SIFT

`cv2.SIFT_create(nfeatures=500)` trên 10 ảnh train mỗi lớp, để xem số keypoint trung bình trước khi chạy toàn bộ.

In [ ]:
run_step("s02_sift_probe.py", 10)

In [ ]:
from IPython.display import Image, display
probe = pd.read_csv(config.OUT / "sift_probe.csv")
display(probe.groupby("label").n_kp.agg(["count", "mean", "min", "max"]).round(1))
Image(str(config.OUT / "sift_probe.png"))

**Nhận xét (lần chạy thử):**

- Trung bình ~437 keypoint/ảnh, descriptor 128 chiều → 700 ảnh train cho ~300k descriptor, lấy mẫu 150k cho k-means là đủ.
- `airplanes` ít keypoint hơn hẳn (~230) vì nền trời trơn → giao dịch ngắn hơn, cần lưu ý khi chọn minsup theo lớp.
- Hầu hết các lớp chạm trần 500 → giữ `nfeatures=500` để độ dài giao dịch giữa các lớp tương đương.
- **Thiên lệch nền:** `Faces_easy` có nhiều keypoint trên kệ sách phía sau, `Leopards` có cỏ và viền đen. Sẽ phân tích ở bước vẽ mẫu.

## 2. Trích SIFT toàn bộ ảnh

Chạy `cv2.SIFT_create(nfeatures=500)` trên cả 1000 ảnh (train và test), lưu **descriptor** (128 chiều) và **tọa độ keypoint** (x, y, size, angle) vào `data/sift.npz`.

- **Vì sao lưu cả test:** ở bước 3 mọi keypoint (train và test) đều được gán từ thị giác bằng codebook; codebook chỉ học từ train nên không rò rỉ.
- **Vì sao lưu tọa độ:** bước vẽ mẫu lên ảnh gốc và mã hóa vị trí lưới 2×2 cần biết keypoint nằm ở đâu.
- **Cách lưu:** một mảng phẳng + `offsets` (keypoint của ảnh *i* nằm ở `[offsets[i], offsets[i+1])`), thứ tự ảnh = thứ tự dòng của `data/split.csv`. Descriptor của OpenCV vốn là số nguyên 0–255 nên lưu `uint8` không mất thông tin (nhỏ hơn 4 lần, file ~49 MB).
- SIFT là thuật toán xác định, không có bước ngẫu nhiên nên chạy lại cho kết quả giống hệt.

In [ ]:
run_step("s03_sift_extract.py")

In [ ]:
import numpy as np
sift_stats = pd.read_csv(config.OUT / "sift_stats.csv")
z = np.load(config.DATA_SIFT)
is_train = (sift_stats.split == "train").to_numpy()
n_kp = np.diff(z["offsets"])
print(f"Descriptor: {z['desc'].shape}, train {n_kp[is_train].sum():,} / test {n_kp[~is_train].sum():,} keypoint")
sift_stats.pivot_table(index="label", columns="split", values="n_kp", aggfunc="mean").round(1).loc[config.CLASSES]

**Nhận xét:**

- 1000 ảnh cho ~450k keypoint, trong đó **~316k descriptor thuộc train** → bước 3 lấy mẫu khoảng 150k descriptor (một nửa) cho k-means là hợp lý, đủ phủ mà không tốn thời gian.
- Khác lần chạy thử 10 ảnh (airplanes ~230), toàn bộ train cho airplanes trung bình ~282 keypoint/ảnh; vẫn thấp hơn hẳn 4 lớp còn lại (~480–500, gần chạm trần 500). Giao dịch của airplanes sẽ **ngắn hơn** → cùng một minsup phần trăm nhưng mật độ khác, cần xem ở bảng thống kê bước 4.
- Không ảnh nào thiếu keypoint (ít nhất 75 keypoint), nên không phải loại ảnh nào.